# 🎵 VIDEOCU Music Studio - Interactive Colab GPU Backend

Bu notebook, VIDEOCU Hit Song Factory sisteminin **interaktif GPU** tarafını çalıştırır.

## ✨ Özellikler
- **Basic Pitch**: Melodi → MIDI dönüşümü (mırıldanma/ıslık algılama)
- **MusicGen**: AI ile profesyonel müzik üretimi
- **Demucs**: Vokal/Enstrüman ayırma
- **İnteraktif Arayüz**: ipywidgets ile gerçek zamanlı kontrol
- **Gradio API**: Telegram/Web entegrasyonu için public URL

## 🎯 Kullanım Modları
1. **İnteraktif Mod**: Adım adım widget'larla çalış
2. **API Mod**: `start_server()` ile Telegram bot'a bağlan
3. **Demo Mod**: `launch_gradio()` ile public demo aç

---
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Cyhn85/videocu-music-studio/blob/main/notebooks/videocu_music_studio.ipynb)

## 1️⃣ GPU Kontrolü ve Kurulum

In [ ]:
# GPU kontrolü
import torch
print(f"🖥️ GPU Mevcut: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ GPU bulunamadı! Runtime → Change runtime type → GPU seçin")

In [ ]:
# Gerekli paketleri kur
!pip install -q basic-pitch
!pip install -q audiocraft
!pip install -q demucs
!pip install -q flask flask-ngrok pyngrok
!pip install -q librosa soundfile
!pip install -q pretty_midi
!pip install -q ipywidgets
!pip install -q gradio

# ipywidgets için Colab desteği
from google.colab import output
output.enable_custom_widget_manager()

print("✅ Tüm paketler kuruldu!")

In [ ]:
# @title 🎤 **İnteraktif Ses Yükleme ve Kontrol Paneli**
# @markdown Bu hücreyi çalıştırarak interaktif arayüzü başlatın

import ipywidgets as widgets
from IPython.display import display, Audio, HTML, clear_output
import io
import os
import tempfile
import numpy as np

# Global state
current_audio_path = None
current_result = None

# === SES YÜKLEME WİDGET'I ===
upload_widget = widgets.FileUpload(
    accept='.wav,.mp3,.ogg,.m4a,.flac',
    multiple=False,
    description='🎤 Ses Seç',
    button_style='primary',
    layout=widgets.Layout(width='200px')
)

# URL'den yükleme
url_input = widgets.Text(
    placeholder='https://example.com/audio.wav',
    description='🌐 URL:',
    layout=widgets.Layout(width='400px')
)

url_load_btn = widgets.Button(
    description='URL\'den Yükle',
    button_style='info',
    icon='download'
)

# Mikrofon kaydı (Colab'da çalışır)
record_btn = widgets.Button(
    description='🎙️ Mikrofon',
    button_style='warning',
    tooltip='Colab\'da mikrofon kaydı'
)

# Durum çıktısı
status_output = widgets.Output()
audio_output = widgets.Output()

def on_upload(change):
    """Dosya yüklendiğinde"""
    global current_audio_path
    
    if upload_widget.value:
        with status_output:
            clear_output()
            file_info = list(upload_widget.value.values())[0]
            filename = list(upload_widget.value.keys())[0]
            content = file_info['content']
            
            # Geçici dosyaya kaydet
            current_audio_path = f"/content/uploaded_{filename}"
            with open(current_audio_path, 'wb') as f:
                f.write(content)
            
            print(f"✅ Yüklendi: {filename}")
            print(f"📁 Boyut: {len(content) / 1024:.1f} KB")
            
        with audio_output:
            clear_output()
            display(Audio(current_audio_path))

def on_url_load(b):
    """URL'den ses yükle"""
    global current_audio_path
    
    url = url_input.value.strip()
    if not url:
        return
    
    with status_output:
        clear_output()
        print(f"⏳ İndiriliyor: {url[:50]}...")
        
        try:
            import urllib.request
            current_audio_path = "/content/url_audio.wav"
            urllib.request.urlretrieve(url, current_audio_path)
            print(f"✅ İndirildi!")
            
            with audio_output:
                clear_output()
                display(Audio(current_audio_path))
        except Exception as e:
            print(f"❌ Hata: {e}")

def on_record(b):
    """Mikrofon kaydı başlat"""
    global current_audio_path
    
    with status_output:
        clear_output()
        print("🎙️ Mikrofon kaydı başlatılıyor...")
        
        # Colab JavaScript kaydı
        from google.colab import output as colab_output
        from base64 import b64decode
        from google.colab.output import eval_js
        
        RECORD_JS = '''
        const sleep = time => new Promise(resolve => setTimeout(resolve, time))
        const b2text = blob => new Promise(resolve => {
          const reader = new FileReader()
          reader.onloadend = e => resolve(e.srcElement.result)
          reader.readAsDataURL(blob)
        })
        
        var record = time => new Promise(async resolve => {
          const stream = await navigator.mediaDevices.getUserMedia({ audio: true })
          const recorder = new MediaRecorder(stream)
          const chunks = []
          recorder.ondataavailable = e => chunks.push(e.data)
          recorder.start()
          await sleep(time)
          recorder.onstop = async () => {
            const blob = new Blob(chunks)
            const text = await b2text(blob)
            resolve(text)
          }
          recorder.stop()
        })
        '''
        
        print("🔴 5 saniye kayıt yapılacak... Şimdi söyleyin!")
        display(colab_output.Javascript(RECORD_JS))
        
        try:
            audio_data = eval_js('record(5000)')
            if audio_data:
                audio_bytes = b64decode(audio_data.split(',')[1])
                current_audio_path = "/content/recorded_audio.wav"
                with open(current_audio_path, 'wb') as f:
                    f.write(audio_bytes)
                print("✅ Kayıt tamamlandı!")
                
                with audio_output:
                    clear_output()
                    display(Audio(current_audio_path))
        except Exception as e:
            print(f"⚠️ Kayıt hatası (tarayıcı izni gerekli): {e}")

# Event handlers
upload_widget.observe(on_upload, names='value')
url_load_btn.on_click(on_url_load)
record_btn.on_click(on_record)

# Layout
upload_box = widgets.VBox([
    widgets.HTML("<h3>📂 Ses Kaynağı Seçin</h3>"),
    widgets.HBox([upload_widget, record_btn]),
    widgets.HBox([url_input, url_load_btn]),
    widgets.HTML("<hr>"),
    status_output,
    audio_output
])

display(upload_box)
print("\n💡 Bir ses dosyası yükleyin, URL girin veya mikrofon kullanın.")

## 2️⃣ Basic Pitch - Melodi Çıkarma

In [ ]:
# @title 🎹 **İnteraktif Melodi Çıkarma ve Piano Roll**
# @markdown Yüklenen sesten melodi çıkar ve görselleştir

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import matplotlib.patches as mpatches

# === İŞLEM SEÇİMİ WİDGET'LARI ===
operation_dropdown = widgets.Dropdown(
    options=[
        ('🎵 Melodi Çıkar (Humming → MIDI)', 'extract_melody'),
        ('🎹 AI Müzik Üret (MusicGen)', 'generate_music'),
        ('🎨 Melodi Dönüştür (Stil Transfer)', 'transform_melody'),
        ('🔀 Vokal/Enstrüman Ayır (Demucs)', 'separate_stems'),
    ],
    value='extract_melody',
    description='İşlem:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='400px')
)

# Stil seçimi (generate/transform için)
style_dropdown = widgets.Dropdown(
    options=[
        ('🎤 Hip-Hop / Trap', 'modern hip hop beat with heavy 808 bass, trap hi-hats'),
        ('🎸 Pop', 'catchy pop melody with bright synths, upbeat drums'),
        ('🎹 Electronic / EDM', 'electronic dance music with pulsing synths, four-on-the-floor beat'),
        ('🎻 Orchestral / Cinematic', 'epic orchestral piece with strings, brass, dramatic percussion'),
        ('☕ Lo-Fi / Chill', 'chill lofi hip hop with jazzy piano, vinyl crackle, relaxed drums'),
        ('🎷 Jazz', 'jazz composition with piano, upright bass, swing drums'),
        ('🎸 Rock', 'rock song with electric guitars, driving drums, powerful bass'),
        ('💿 R&B / Soul', 'smooth r&b with soulful chords, soft drums, warm bass'),
    ],
    value='modern hip hop beat with heavy 808 bass, trap hi-hats',
    description='Stil:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='400px')
)

# Süre slider
duration_slider = widgets.IntSlider(
    value=30,
    min=5,
    max=120,
    step=5,
    description='Süre (s):',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='400px')
)

# İşlem butonu
process_btn = widgets.Button(
    description='🚀 İşlemi Başlat',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)

# Progress bar
progress = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description='İlerleme:',
    bar_style='info',
    layout=widgets.Layout(width='400px')
)

# Sonuç çıktıları
result_output = widgets.Output()
piano_roll_output = widgets.Output()

def show_piano_roll(notes_info, title="🎹 Melodi Piano Roll"):
    """Notaları piano roll olarak görselleştir"""
    with piano_roll_output:
        clear_output()
        
        if not notes_info:
            print("❌ Görselleştirilecek nota yok")
            return
        
        fig, ax = plt.subplots(figsize=(14, 6))
        
        # Renk paleti (oktava göre)
        colors = plt.cm.viridis(np.linspace(0.2, 0.8, 12))
        
        for note in notes_info:
            pitch = note['pitch']
            start = note['start']
            duration = note['end'] - note['start']
            
            # Oktava göre renk
            color = colors[pitch % 12]
            
            rect = Rectangle(
                (start, pitch - 0.4),
                duration,
                0.8,
                facecolor=color,
                edgecolor='black',
                linewidth=0.5,
                alpha=0.8
            )
            ax.add_patch(rect)
        
        # Eksen ayarları
        all_pitches = [n['pitch'] for n in notes_info]
        all_times = [n['end'] for n in notes_info]
        
        ax.set_xlim(0, max(all_times) + 0.5)
        ax.set_ylim(min(all_pitches) - 2, max(all_pitches) + 2)
        
        ax.set_xlabel('Zaman (saniye)', fontsize=12)
        ax.set_ylabel('MIDI Nota Numarası', fontsize=12)
        ax.set_title(title, fontsize=14, fontweight='bold')
        
        # Y ekseni nota isimleri
        note_names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
        yticks = range(min(all_pitches), max(all_pitches) + 1)
        yticklabels = [f"{note_names[p % 12]}{p // 12 - 1}" for p in yticks]
        ax.set_yticks(yticks)
        ax.set_yticklabels(yticklabels, fontsize=8)
        
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

def on_process(b):
    """İşlem butonuna tıklandığında"""
    global current_result
    
    operation = operation_dropdown.value
    
    with result_output:
        clear_output()
        
        if operation in ['extract_melody', 'transform_melody', 'separate_stems']:
            if current_audio_path is None:
                print("❌ Önce bir ses dosyası yükleyin!")
                return
        
        progress.value = 10
        
        if operation == 'extract_melody':
            print("🎵 Melodi çıkarılıyor (Basic Pitch)...")
            progress.value = 30
            
            try:
                result = melody_extractor.extract_melody(current_audio_path)
                current_result = result
                progress.value = 80
                
                print("\n" + "="*50)
                print("✅ MELODİ BAŞARIYLA ÇIKARILDI!")
                print("="*50)
                print(f"🎵 Nota Sayısı: {result['note_count']}")
                print(f"🎼 Anahtar: {result['key']}")
                print(f"⏱️ Tempo: {result['tempo']} BPM")
                print(f"⏳ Süre: {result['duration']:.1f} saniye")
                print(f"💾 MIDI: {result['midi_path']}")
                
                # Piano roll göster
                show_piano_roll(result['notes'], f"🎹 Melodi - {result['note_count']} nota, {result['key']}")
                
                progress.value = 100
                
            except Exception as e:
                print(f"❌ Hata: {e}")
                progress.value = 0
        
        elif operation == 'generate_music':
            prompt = style_dropdown.value
            duration = duration_slider.value
            
            print(f"🎹 Müzik üretiliyor...")
            print(f"   Stil: {prompt[:50]}...")
            print(f"   Süre: {duration}s")
            progress.value = 30
            
            try:
                output_path = music_generator.generate_from_prompt(prompt, duration)
                progress.value = 90
                
                print("\n✅ Müzik üretildi!")
                display(Audio(output_path))
                progress.value = 100
                
            except Exception as e:
                print(f"❌ Hata: {e}")
                progress.value = 0
        
        elif operation == 'transform_melody':
            prompt = style_dropdown.value
            duration = duration_slider.value
            
            print(f"🎨 Melodi dönüştürülüyor...")
            progress.value = 30
            
            try:
                output_path = music_generator.generate_continuation(
                    current_audio_path, prompt, duration
                )
                progress.value = 90
                
                print("\n✅ Dönüştürme tamamlandı!")
                display(Audio(output_path))
                progress.value = 100
                
            except Exception as e:
                print(f"❌ Hata: {e}")
                progress.value = 0
        
        elif operation == 'separate_stems':
            print("🔀 Stem'ler ayrılıyor (Demucs)...")
            progress.value = 30
            
            try:
                result = stem_separator.separate(current_audio_path)
                progress.value = 90
                
                print("\n✅ Stem ayırma tamamlandı!")
                print("\n🎤 Vokal:")
                display(Audio(result['vocals']))
                print("\n🎸 Enstrümental:")
                display(Audio(result['no_vocals']))
                progress.value = 100
                
            except Exception as e:
                print(f"❌ Hata: {e}")
                progress.value = 0

process_btn.on_click(on_process)

# İşlem seçimine göre widget görünürlüğü
def on_operation_change(change):
    op = change['new']
    if op in ['generate_music', 'transform_melody']:
        style_dropdown.layout.display = 'flex'
        duration_slider.layout.display = 'flex'
    else:
        style_dropdown.layout.display = 'none'
        duration_slider.layout.display = 'none'

operation_dropdown.observe(on_operation_change, names='value')
# Başlangıçta gizle
style_dropdown.layout.display = 'none'
duration_slider.layout.display = 'none'

# Layout
process_box = widgets.VBox([
    widgets.HTML("<h3>⚙️ İşlem Seçin</h3>"),
    operation_dropdown,
    style_dropdown,
    duration_slider,
    widgets.HBox([process_btn, progress]),
    widgets.HTML("<hr>"),
    result_output,
    piano_roll_output
])

display(process_box)

In [ ]:
from basic_pitch.inference import predict
from basic_pitch import ICASSP_2022_MODEL_PATH
import pretty_midi
import numpy as np
import os

class MelodyExtractor:
    """Basic Pitch ile melodi çıkarma"""
    
    def __init__(self):
        print("🎵 Basic Pitch modeli yükleniyor...")
        self.model_path = ICASSP_2022_MODEL_PATH
        print("✅ Model hazır!")
    
    def extract_melody(self, audio_path, output_dir="/content/output"):
        """
        Ses dosyasından melodi çıkar ve MIDI'ye dönüştür
        
        Args:
            audio_path: Girdi ses dosyası
            output_dir: Çıktı klasörü
            
        Returns:
            dict: MIDI bilgileri ve dosya yolu
        """
        os.makedirs(output_dir, exist_ok=True)
        
        print(f"🎤 Melodi çıkarılıyor: {audio_path}")
        
        # Basic Pitch ile tahmin
        model_output, midi_data, note_events = predict(
            audio_path,
            self.model_path
        )
        
        # MIDI dosyasını kaydet
        base_name = os.path.splitext(os.path.basename(audio_path))[0]
        midi_path = os.path.join(output_dir, f"{base_name}_melody.mid")
        midi_data.write(midi_path)
        
        # Nota bilgilerini çıkar
        notes_info = []
        for note in midi_data.instruments[0].notes:
            notes_info.append({
                "pitch": note.pitch,
                "start": round(note.start, 3),
                "end": round(note.end, 3),
                "velocity": note.velocity,
                "note_name": pretty_midi.note_number_to_name(note.pitch)
            })
        
        # Anahtar ve tempo tahmini
        key_estimate = self._estimate_key(notes_info)
        tempo_estimate = self._estimate_tempo(notes_info)
        
        result = {
            "midi_path": midi_path,
            "note_count": len(notes_info),
            "notes": notes_info,
            "duration": max([n["end"] for n in notes_info]) if notes_info else 0,
            "key": key_estimate,
            "tempo": tempo_estimate
        }
        
        print(f"✅ Melodi çıkarıldı!")
        print(f"   - Nota sayısı: {result['note_count']}")
        print(f"   - Süre: {result['duration']:.1f}s")
        print(f"   - Anahtar: {result['key']}")
        print(f"   - Tempo: {result['tempo']} BPM")
        
        return result
    
    def _estimate_key(self, notes):
        """Basit anahtar tahmini"""
        if not notes:
            return "C major"
        
        # En sık kullanılan notaları bul
        pitch_classes = [n["pitch"] % 12 for n in notes]
        from collections import Counter
        common = Counter(pitch_classes).most_common(3)
        
        # Basit major/minor tahmini
        key_names = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
        root = key_names[common[0][0]]
        
        return f"{root} major"
    
    def _estimate_tempo(self, notes):
        """Nota aralıklarından tempo tahmini"""
        if len(notes) < 2:
            return 120
        
        # Nota başlangıç aralıkları
        intervals = []
        for i in range(1, len(notes)):
            interval = notes[i]["start"] - notes[i-1]["start"]
            if 0.1 < interval < 2:  # Makul aralıklar
                intervals.append(interval)
        
        if not intervals:
            return 120
        
        avg_interval = np.median(intervals)
        tempo = int(60 / avg_interval) if avg_interval > 0 else 120
        
        # Makul aralığa sınırla
        return max(60, min(180, tempo))

# Global instance
melody_extractor = MelodyExtractor()

In [ ]:
# @title 🎼 **Nota Düzenleme Widget'ları**
# @markdown Çıkarılan melodinin notalarını düzenleyin

# === NOTA DÜZENLEME WİDGET'LARI ===
note_select = widgets.Dropdown(
    options=[],
    description='Nota:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='300px')
)

pitch_slider = widgets.IntSlider(
    value=60,
    min=21,
    max=108,
    description='Perde:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='400px')
)

start_slider = widgets.FloatSlider(
    value=0,
    min=0,
    max=60,
    step=0.05,
    description='Başlangıç:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='400px')
)

duration_edit_slider = widgets.FloatSlider(
    value=0.5,
    min=0.05,
    max=5,
    step=0.05,
    description='Süre:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='400px')
)

# Nota adı gösterimi
pitch_label = widgets.HTML(value="<b>Nota: C4</b>")

# Düzenleme butonları
update_note_btn = widgets.Button(
    description='✏️ Notayı Güncelle',
    button_style='warning',
    layout=widgets.Layout(width='150px')
)

delete_note_btn = widgets.Button(
    description='🗑️ Notayı Sil',
    button_style='danger',
    layout=widgets.Layout(width='150px')
)

add_note_btn = widgets.Button(
    description='➕ Yeni Nota Ekle',
    button_style='success',
    layout=widgets.Layout(width='150px')
)

refresh_btn = widgets.Button(
    description='🔄 Görselleştir',
    button_style='info',
    layout=widgets.Layout(width='150px')
)

edit_output = widgets.Output()
edit_piano_roll = widgets.Output()

def pitch_to_name(pitch):
    """MIDI pitch -> nota adı"""
    note_names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
    return f"{note_names[pitch % 12]}{pitch // 12 - 1}"

def update_note_list():
    """Nota listesini güncelle"""
    global current_result
    if current_result and current_result.get('notes'):
        options = []
        for i, note in enumerate(current_result['notes']):
            name = pitch_to_name(note['pitch'])
            options.append((f"{i+1}. {name} ({note['start']:.2f}s - {note['end']:.2f}s)", i))
        note_select.options = options
        if options:
            note_select.value = 0

def on_note_select(change):
    """Nota seçildiğinde slider'ları güncelle"""
    global current_result
    if current_result and current_result.get('notes') and change['new'] is not None:
        note = current_result['notes'][change['new']]
        pitch_slider.value = note['pitch']
        start_slider.value = note['start']
        duration_edit_slider.value = note['end'] - note['start']
        pitch_label.value = f"<b>Nota: {pitch_to_name(note['pitch'])}</b>"

def on_pitch_change(change):
    """Pitch değiştiğinde nota adını güncelle"""
    pitch_label.value = f"<b>Nota: {pitch_to_name(change['new'])}</b>"

def on_update_note(b):
    """Notayı güncelle"""
    global current_result
    if current_result and note_select.value is not None:
        idx = note_select.value
        current_result['notes'][idx]['pitch'] = pitch_slider.value
        current_result['notes'][idx]['start'] = start_slider.value
        current_result['notes'][idx]['end'] = start_slider.value + duration_edit_slider.value
        
        with edit_output:
            clear_output()
            print(f"✅ Nota {idx+1} güncellendi!")
        
        update_note_list()
        on_refresh(None)

def on_delete_note(b):
    """Notayı sil"""
    global current_result
    if current_result and note_select.value is not None:
        idx = note_select.value
        del current_result['notes'][idx]
        current_result['note_count'] = len(current_result['notes'])
        
        with edit_output:
            clear_output()
            print(f"🗑️ Nota silindi!")
        
        update_note_list()
        on_refresh(None)

def on_add_note(b):
    """Yeni nota ekle"""
    global current_result
    if current_result:
        new_note = {
            'pitch': pitch_slider.value,
            'start': start_slider.value,
            'end': start_slider.value + duration_edit_slider.value,
            'velocity': 100,
            'note_name': pitch_to_name(pitch_slider.value)
        }
        current_result['notes'].append(new_note)
        current_result['notes'].sort(key=lambda x: x['start'])
        current_result['note_count'] = len(current_result['notes'])
        
        with edit_output:
            clear_output()
            print(f"➕ Yeni nota eklendi: {pitch_to_name(pitch_slider.value)}")
        
        update_note_list()
        on_refresh(None)

def on_refresh(b):
    """Piano roll'u yenile"""
    global current_result
    if current_result and current_result.get('notes'):
        with edit_piano_roll:
            clear_output()
            
            fig, ax = plt.subplots(figsize=(14, 5))
            colors = plt.cm.viridis(np.linspace(0.2, 0.8, 12))
            
            for i, note in enumerate(current_result['notes']):
                color = colors[note['pitch'] % 12]
                is_selected = (note_select.value == i)
                
                rect = Rectangle(
                    (note['start'], note['pitch'] - 0.4),
                    note['end'] - note['start'],
                    0.8,
                    facecolor=color,
                    edgecolor='red' if is_selected else 'black',
                    linewidth=3 if is_selected else 0.5,
                    alpha=0.9 if is_selected else 0.7
                )
                ax.add_patch(rect)
            
            all_pitches = [n['pitch'] for n in current_result['notes']]
            all_times = [n['end'] for n in current_result['notes']]
            
            ax.set_xlim(0, max(all_times) + 0.5)
            ax.set_ylim(min(all_pitches) - 2, max(all_pitches) + 2)
            ax.set_xlabel('Zaman (saniye)')
            ax.set_ylabel('MIDI Nota')
            ax.set_title(f"🎼 Düzenlenebilir Piano Roll - {len(current_result['notes'])} nota")
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()

# Event handlers
note_select.observe(on_note_select, names='value')
pitch_slider.observe(on_pitch_change, names='value')
update_note_btn.on_click(on_update_note)
delete_note_btn.on_click(on_delete_note)
add_note_btn.on_click(on_add_note)
refresh_btn.on_click(on_refresh)

# Layout
edit_box = widgets.VBox([
    widgets.HTML("<h3>🎼 Nota Düzenleme</h3>"),
    widgets.HTML("<p><i>Önce yukarıdan melodi çıkarın, sonra burada düzenleyin.</i></p>"),
    note_select,
    widgets.HBox([pitch_slider, pitch_label]),
    start_slider,
    duration_edit_slider,
    widgets.HBox([update_note_btn, delete_note_btn, add_note_btn, refresh_btn]),
    edit_output,
    edit_piano_roll
])

display(edit_box)
print("💡 Melodi çıkardıktan sonra notaları düzenleyebilirsiniz.")

## 3️⃣ MusicGen - AI Müzik Üretimi

In [ ]:
from audiocraft.models import MusicGen
from audiocraft.data.audio import audio_write
import torchaudio

class MusicGenerator:
    """MusicGen ile AI müzik üretimi"""
    
    def __init__(self, model_size="small"):
        """
        Args:
            model_size: 'small', 'medium', 'large' (GPU'ya göre seç)
        """
        print(f"🎹 MusicGen {model_size} modeli yükleniyor...")
        self.model = MusicGen.get_pretrained(f'facebook/musicgen-{model_size}')
        self.model.set_generation_params(
            duration=30,  # Varsayılan süre
            temperature=1.0,
            top_k=250,
            top_p=0.0
        )
        print("✅ MusicGen hazır!")
    
    def generate_from_prompt(self, prompt, duration=30, output_path="/content/output/generated.wav"):
        """
        Metin prompt'undan müzik üret
        
        Args:
            prompt: Stil açıklaması (örn: "upbeat hip-hop beat with 808 bass")
            duration: Süre (saniye)
            output_path: Çıktı dosyası
        """
        print(f"🎵 Müzik üretiliyor: {prompt[:50]}...")
        
        self.model.set_generation_params(duration=duration)
        
        # Üret
        wav = self.model.generate([prompt])
        
        # Kaydet
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        audio_write(
            output_path.replace('.wav', ''),
            wav[0].cpu(),
            self.model.sample_rate,
            strategy="loudness"
        )
        
        print(f"✅ Müzik üretildi: {output_path}")
        return output_path + ".wav"
    
    def generate_continuation(self, audio_path, prompt, duration=30, output_path="/content/output/continued.wav"):
        """
        Mevcut sesi devam ettir/dönüştür
        
        Args:
            audio_path: Referans ses dosyası
            prompt: Stil açıklaması
            duration: Toplam süre
        """
        print(f"🎵 Ses dönüştürülüyor: {audio_path}")
        
        # Referans sesi yükle
        melody_waveform, sr = torchaudio.load(audio_path)
        
        # Mono'ya çevir
        if melody_waveform.shape[0] > 1:
            melody_waveform = melody_waveform.mean(dim=0, keepdim=True)
        
        # Sample rate'i ayarla
        if sr != self.model.sample_rate:
            resampler = torchaudio.transforms.Resample(sr, self.model.sample_rate)
            melody_waveform = resampler(melody_waveform)
        
        self.model.set_generation_params(duration=duration)
        
        # Melody-conditioned generation
        wav = self.model.generate_with_chroma(
            descriptions=[prompt],
            melody_wavs=melody_waveform.unsqueeze(0).cuda(),
            melody_sample_rate=self.model.sample_rate,
            progress=True
        )
        
        # Kaydet
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        audio_write(
            output_path.replace('.wav', ''),
            wav[0].cpu(),
            self.model.sample_rate,
            strategy="loudness"
        )
        
        print(f"✅ Dönüştürülmüş ses: {output_path}")
        return output_path + ".wav"

# Model yükle (T4 için small, V100 için medium önerilir)
print("MusicGen modeli yüklenecek. Bu biraz zaman alabilir...")
music_generator = MusicGenerator("small")

## 4️⃣ Demucs - Stem Separation

In [ ]:
import demucs.separate
import subprocess

class StemSeparator:
    """Demucs ile stem ayırma"""
    
    def separate(self, audio_path, output_dir="/content/output/stems"):
        """
        Şarkıyı stem'lerine ayır (vocals, drums, bass, other)
        
        Args:
            audio_path: Girdi ses dosyası
            output_dir: Çıktı klasörü
        """
        print(f"🎛️ Stem ayırma başlıyor: {audio_path}")
        
        os.makedirs(output_dir, exist_ok=True)
        
        # Demucs komutunu çalıştır
        cmd = f"python -m demucs -o {output_dir} --two-stems vocals {audio_path}"
        subprocess.run(cmd.split(), check=True)
        
        # Çıktı dosyalarını bul
        base_name = os.path.splitext(os.path.basename(audio_path))[0]
        stems_dir = os.path.join(output_dir, "htdemucs", base_name)
        
        result = {
            "vocals": os.path.join(stems_dir, "vocals.wav"),
            "no_vocals": os.path.join(stems_dir, "no_vocals.wav")
        }
        
        print("✅ Stem ayırma tamamlandı!")
        print(f"   - Vocals: {result['vocals']}")
        print(f"   - Instrumental: {result['no_vocals']}")
        
        return result

stem_separator = StemSeparator()
print("✅ Stem Separator hazır!")

## 5️⃣ API Server - CPU Sunucusuyla İletişim

In [ ]:
from flask import Flask, request, jsonify, send_file
from pyngrok import ngrok
import base64
import io
import tempfile
import threading

app = Flask(__name__)

# Ngrok auth token'ınızı buraya girin
NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN_HERE"  # https://dashboard.ngrok.com/get-started/your-authtoken

@app.route('/health', methods=['GET'])
def health():
    """Sağlık kontrolü"""
    return jsonify({
        "status": "healthy",
        "gpu": torch.cuda.is_available(),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    })

@app.route('/extract-melody', methods=['POST'])
def api_extract_melody():
    """Melodi çıkarma endpoint'i"""
    try:
        # Base64 encoded audio al
        data = request.json
        audio_b64 = data.get('audio')
        filename = data.get('filename', 'input.wav')
        
        # Geçici dosyaya kaydet
        audio_bytes = base64.b64decode(audio_b64)
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
            f.write(audio_bytes)
            temp_path = f.name
        
        # Melodi çıkar
        result = melody_extractor.extract_melody(temp_path)
        
        # MIDI dosyasını base64'e çevir
        with open(result['midi_path'], 'rb') as f:
            midi_b64 = base64.b64encode(f.read()).decode()
        
        result['midi_base64'] = midi_b64
        del result['midi_path']  # Yerel yolu kaldır
        
        return jsonify({"success": True, "result": result})
        
    except Exception as e:
        return jsonify({"success": False, "error": str(e)}), 500

@app.route('/generate-music', methods=['POST'])
def api_generate_music():
    """MusicGen ile müzik üretme"""
    try:
        data = request.json
        prompt = data.get('prompt', 'upbeat electronic music')
        duration = data.get('duration', 30)
        
        # Üret
        output_path = music_generator.generate_from_prompt(prompt, duration)
        
        # Base64'e çevir
        with open(output_path, 'rb') as f:
            audio_b64 = base64.b64encode(f.read()).decode()
        
        return jsonify({
            "success": True,
            "audio_base64": audio_b64,
            "duration": duration
        })
        
    except Exception as e:
        return jsonify({"success": False, "error": str(e)}), 500

@app.route('/transform-melody', methods=['POST'])
def api_transform_melody():
    """Melodiyi profesyonel sese dönüştür"""
    try:
        data = request.json
        audio_b64 = data.get('audio')
        prompt = data.get('prompt', 'professional pop melody with synth')
        duration = data.get('duration', 30)
        
        # Geçici dosyaya kaydet
        audio_bytes = base64.b64decode(audio_b64)
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
            f.write(audio_bytes)
            temp_path = f.name
        
        # Dönüştür
        output_path = music_generator.generate_continuation(
            temp_path, prompt, duration
        )
        
        # Base64'e çevir
        with open(output_path, 'rb') as f:
            audio_b64 = base64.b64encode(f.read()).decode()
        
        return jsonify({
            "success": True,
            "audio_base64": audio_b64
        })
        
    except Exception as e:
        return jsonify({"success": False, "error": str(e)}), 500

@app.route('/separate-stems', methods=['POST'])
def api_separate_stems():
    """Stem ayırma"""
    try:
        data = request.json
        audio_b64 = data.get('audio')
        
        # Geçici dosyaya kaydet
        audio_bytes = base64.b64decode(audio_b64)
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
            f.write(audio_bytes)
            temp_path = f.name
        
        # Ayır
        result = stem_separator.separate(temp_path)
        
        # Base64'e çevir
        stems_b64 = {}
        for stem_name, stem_path in result.items():
            if os.path.exists(stem_path):
                with open(stem_path, 'rb') as f:
                    stems_b64[stem_name] = base64.b64encode(f.read()).decode()
        
        return jsonify({
            "success": True,
            "stems": stems_b64
        })
        
    except Exception as e:
        return jsonify({"success": False, "error": str(e)}), 500

def start_server():
    """API sunucusunu başlat"""
    # Ngrok bağlantısı
    if NGROK_AUTH_TOKEN != "YOUR_NGROK_TOKEN_HERE":
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    
    public_url = ngrok.connect(5000)
    print(f"\n🌐 API Sunucusu hazır!")
    print(f"📡 Public URL: {public_url}")
    print(f"\n💡 Bu URL'i CPU sunucusuna kaydedin!")
    print(f"\nEndpoints:")
    print(f"  GET  /health           - Sağlık kontrolü")
    print(f"  POST /extract-melody   - Melodi çıkar")
    print(f"  POST /generate-music   - Müzik üret")
    print(f"  POST /transform-melody - Melodiyi dönüştür")
    print(f"  POST /separate-stems   - Stem ayır")
    
    # Flask'ı arka planda çalıştır
    app.run(port=5000)

In [ ]:
# @title 🌐 **Gradio API - Telegram/Web Entegrasyonu (Opsiyonel)**
# @markdown Bu hücreyi çalıştırarak public API URL'i oluşturun

import gradio as gr

def gradio_extract_melody(audio_file):
    """Gradio için melodi çıkarma"""
    if audio_file is None:
        return None, {"error": "Ses dosyası gerekli"}
    
    result = melody_extractor.extract_melody(audio_file)
    
    # Piano roll görselini oluştur
    import matplotlib
    matplotlib.use('Agg')
    fig, ax = plt.subplots(figsize=(12, 4))
    
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, 12))
    for note in result['notes']:
        color = colors[note['pitch'] % 12]
        rect = Rectangle(
            (note['start'], note['pitch'] - 0.4),
            note['end'] - note['start'],
            0.8,
            facecolor=color,
            edgecolor='black',
            linewidth=0.5,
            alpha=0.8
        )
        ax.add_patch(rect)
    
    if result['notes']:
        all_pitches = [n['pitch'] for n in result['notes']]
        all_times = [n['end'] for n in result['notes']]
        ax.set_xlim(0, max(all_times) + 0.5)
        ax.set_ylim(min(all_pitches) - 2, max(all_pitches) + 2)
    
    ax.set_xlabel('Zaman (s)')
    ax.set_ylabel('MIDI Nota')
    ax.set_title(f"🎹 {result['note_count']} nota - {result['key']} - {result['tempo']} BPM")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    
    return fig, {
        "nota_sayisi": result['note_count'],
        "anahtar": result['key'],
        "tempo": result['tempo'],
        "sure": f"{result['duration']:.1f}s",
        "midi_dosyasi": result['midi_path']
    }

def gradio_generate_music(style, custom_prompt, duration):
    """Gradio için müzik üretme"""
    prompt = custom_prompt if custom_prompt.strip() else style
    
    try:
        output_path = music_generator.generate_from_prompt(prompt, int(duration))
        return output_path
    except Exception as e:
        return None

def gradio_transform_melody(audio_file, style, duration):
    """Gradio için melodi dönüştürme"""
    if audio_file is None:
        return None
    
    try:
        output_path = music_generator.generate_continuation(
            audio_file, style, int(duration)
        )
        return output_path
    except Exception as e:
        return None

def gradio_separate_stems(audio_file):
    """Gradio için stem ayırma"""
    if audio_file is None:
        return None, None
    
    try:
        result = stem_separator.separate(audio_file)
        return result['vocals'], result['no_vocals']
    except Exception as e:
        return None, None

# Gradio arayüzü oluştur
with gr.Blocks(title="🎵 VIDEOCU Music Studio", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🎵 VIDEOCU Music Studio
    
    **GPU-powered müzik üretimi** - Basic Pitch, MusicGen, Demucs
    
    ---
    """)
    
    with gr.Tabs():
        # Tab 1: Melodi Çıkarma
        with gr.TabItem("🎵 Melodi Çıkar"):
            with gr.Row():
                with gr.Column():
                    melody_input = gr.Audio(
                        label="Ses Dosyası (Mırıldanma/Islık)",
                        type="filepath"
                    )
                    melody_btn = gr.Button("🎵 Melodi Çıkar", variant="primary")
                
                with gr.Column():
                    melody_plot = gr.Plot(label="Piano Roll")
                    melody_info = gr.JSON(label="Melodi Bilgileri")
            
            melody_btn.click(
                gradio_extract_melody,
                inputs=[melody_input],
                outputs=[melody_plot, melody_info]
            )
        
        # Tab 2: AI Müzik Üret
        with gr.TabItem("🎹 AI Müzik Üret"):
            with gr.Row():
                with gr.Column():
                    style_select = gr.Dropdown(
                        choices=[
                            "modern hip hop beat with heavy 808 bass, trap hi-hats",
                            "catchy pop melody with bright synths, upbeat drums",
                            "electronic dance music with pulsing synths",
                            "chill lofi hip hop with jazzy piano, vinyl crackle",
                            "epic orchestral piece with strings and brass",
                            "jazz composition with piano and upright bass"
                        ],
                        label="Stil Seç",
                        value="modern hip hop beat with heavy 808 bass, trap hi-hats"
                    )
                    custom_prompt = gr.Textbox(
                        label="Özel Prompt (opsiyonel)",
                        placeholder="Kendi stilinizi yazın..."
                    )
                    gen_duration = gr.Slider(5, 120, value=30, step=5, label="Süre (saniye)")
                    generate_btn = gr.Button("🎹 Üret", variant="primary")
                
                with gr.Column():
                    generated_audio = gr.Audio(label="Üretilen Müzik")
            
            generate_btn.click(
                gradio_generate_music,
                inputs=[style_select, custom_prompt, gen_duration],
                outputs=[generated_audio]
            )
        
        # Tab 3: Melodi Dönüştür
        with gr.TabItem("🎨 Stil Dönüştür"):
            with gr.Row():
                with gr.Column():
                    transform_input = gr.Audio(
                        label="Kaynak Ses",
                        type="filepath"
                    )
                    transform_style = gr.Dropdown(
                        choices=[
                            "professional pop with synth and bass",
                            "hip hop beat with 808",
                            "orchestral arrangement",
                            "electronic remix"
                        ],
                        label="Hedef Stil",
                        value="professional pop with synth and bass"
                    )
                    transform_duration = gr.Slider(5, 120, value=30, step=5, label="Süre")
                    transform_btn = gr.Button("🎨 Dönüştür", variant="primary")
                
                with gr.Column():
                    transformed_audio = gr.Audio(label="Dönüştürülmüş Ses")
            
            transform_btn.click(
                gradio_transform_melody,
                inputs=[transform_input, transform_style, transform_duration],
                outputs=[transformed_audio]
            )
        
        # Tab 4: Stem Ayır
        with gr.TabItem("🔀 Stem Ayır"):
            with gr.Row():
                with gr.Column():
                    stems_input = gr.Audio(
                        label="Şarkı Dosyası",
                        type="filepath"
                    )
                    stems_btn = gr.Button("🔀 Ayır", variant="primary")
                
                with gr.Column():
                    vocals_output = gr.Audio(label="🎤 Vokal")
                    instrumental_output = gr.Audio(label="🎸 Enstrümental")
            
            stems_btn.click(
                gradio_separate_stems,
                inputs=[stems_input],
                outputs=[vocals_output, instrumental_output]
            )
    
    gr.Markdown("""
    ---
    **💡 İpucu:** Bu URL'i Telegram bot'unuza veya web arayüzünüze entegre edebilirsiniz.
    
    **API Kullanımı:**
    ```python
    from gradio_client import Client
    client = Client("YOUR_GRADIO_URL")
    result = client.predict(audio_file, api_name="/extract_melody")
    ```
    """)

def launch_gradio():
    """Gradio public URL'i başlat"""
    print("🌐 Gradio başlatılıyor...")
    demo.launch(share=True, debug=True)
    
print("✅ Gradio hazır!")
print("\n🚀 Public URL almak için: launch_gradio()")
print("   Bu URL'i Telegram bot'unuza veya web sitenize entegre edebilirsiniz.")

## 6️⃣ Test Fonksiyonları

In [ ]:
# Test: Melodi çıkarma
def test_melody_extraction(audio_url):
    """URL'den ses indirip melodi çıkar"""
    import urllib.request
    
    # İndir
    temp_path = "/content/test_audio.wav"
    urllib.request.urlretrieve(audio_url, temp_path)
    
    # Çıkar
    result = melody_extractor.extract_melody(temp_path)
    return result

# Test: Müzik üretimi
def test_music_generation(prompt="happy electronic music with upbeat drums", duration=15):
    """Prompt'tan müzik üret"""
    output = music_generator.generate_from_prompt(prompt, duration)
    return output

# Test: Melodi dönüştürme
def test_melody_transform(audio_path, style="professional pop with synth and bass"):
    """Melodiyi dönüştür"""
    output = music_generator.generate_continuation(audio_path, style, 30)
    return output

print("✅ Test fonksiyonları hazır!")
print("\nKullanım:")
print('  result = test_music_generation("hip hop beat with 808 bass", 20)')
print('  result = test_melody_extraction("https://example.com/audio.wav")')

## 🚀 Sunucuyu Başlat

In [ ]:
# ⚠️ Bu hücreyi çalıştırmadan önce NGROK_AUTH_TOKEN'ı güncelleyin!
# Token almak için: https://dashboard.ngrok.com/get-started/your-authtoken

# Sunucuyu başlat
start_server()

---

## 📝 Kullanım Kılavuzu

### 1. İlk Kurulum
1. Runtime → Change runtime type → GPU seçin
2. Tüm hücreleri sırayla çalıştırın (Runtime → Run all)
3. ngrok token'ınızı girin (ücretsiz: ngrok.com)

### 2. CPU Sunucusu Bağlantısı
Sunucu başladığında görünen Public URL'i kopyalayın:
```
https://xxxx-xx-xx-xxx-xxx.ngrok.io
```

Bu URL'i CPU sunucusundaki .env dosyasına ekleyin:
```
COLAB_API_URL=https://xxxx-xx-xx-xxx-xxx.ngrok.io
```

### 3. API Kullanımı

**Melodi Çıkarma:**
```python
import requests
import base64

with open('humming.wav', 'rb') as f:
    audio_b64 = base64.b64encode(f.read()).decode()

response = requests.post(
    'https://xxx.ngrok.io/extract-melody',
    json={'audio': audio_b64}
)
result = response.json()
print(f"Nota sayısı: {result['result']['note_count']}")
```

**Müzik Üretme:**
```python
response = requests.post(
    'https://xxx.ngrok.io/generate-music',
    json={
        'prompt': 'energetic hip hop beat with 808 bass and trap hi-hats',
        'duration': 30
    }
)
audio_b64 = response.json()['audio_base64']
```

### 4. Stil Örnekleri
- `"upbeat pop melody with bright synths and catchy rhythm"`
- `"dark trap beat with heavy 808 bass and hi-hats"`
- `"chill lofi hip hop with jazzy piano and vinyl crackle"`
- `"epic orchestral music with strings and brass"`
- `"acoustic guitar ballad with soft drums"`